# Hexagon-style heterogeneous NPU on the PYNQ-Z1

This notebook exercises the units added by issue #136 on the board, one cell at a time:

1. load the `--hexagon` overlay and identify the subsystem `npu_hx_0`;
2. run an RV32I program on the **scalar core** (through the L1 I$/D$ and L2) and check it against numpy and the reference ISS;
3. run INT8 and FP16 kernels on the **vector unit** (DDR -> DMA -> TCM -> vector -> DMA -> DDR) and compare bit-exactly with numpy;
4. **STOP** a program that never ends and resume it;
5. run **SmolLM2** on the same overlay (its GEMMs use the unchanged tensor unit).

Expected layout (what `examples/hexagon-npu/README.md` sets up): this notebook next to `src/` (the repository's), `overlay/npu_matrix.bit` + `.hwh` (built with `build_overlay.tcl --hexagon`) and `export/smollm` (`exp/1011_hexagon_npu/export_smollm.sh`). In a repository checkout it also finds `build/vivado/npu_matrix_16x16_hx/artifacts/`.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

here = Path.cwd().resolve()
root = next((p for p in (here, *here.parents) if (p / 'src' / 'runtime' / 'hexagon.py').is_file()), None)
if root is None:
    raise RuntimeError('start the notebook in a directory that has (or is inside a tree with) src/runtime/hexagon.py')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.runtime.hexagon import HexagonNpu, VectorOp, describe_status
from src.test.model.rv32_asm import assemble
from src.test.model.rv32_iss import RV32ISS

candidates = [here / 'overlay' / 'npu_matrix.bit',
              root / 'overlay' / 'npu_matrix.bit',
              root / 'build' / 'vivado' / 'npu_matrix_16x16_hx' / 'artifacts' / 'npu_matrix.bit']
bitstream = next((p for p in candidates if p.is_file()), None)
if bitstream is None:
    raise FileNotFoundError('no --hexagon bitstream in: ' + ', '.join(map(str, candidates)))
print('repository root:', root)
print('bitstream      :', bitstream)

## 1. Load the overlay and identify the subsystem

`npu_hx_0` (0x43C10000) sits next to the unchanged tensor accelerator `npu_accelerator_0` (0x43C00000). CAPS should read `0x0010087F`: scalar, vector, L1I, L1D, L2, TCM and DMA present, 64-bit vectors, 1024 TCM words.

In [ ]:
from pynq import Overlay

overlay = Overlay(str(bitstream))
npu = HexagonNpu(overlay)
info = npu.info()
info

In [ ]:
assert info['magic'] == '0x314e5848', 'not the Hexagon subsystem'
assert npu.caps.raw == 0x0010087F, hex(npu.caps.raw)
print('IPs in the overlay:', [name for name in overlay.ip_dict if name.startswith('npu')])
print('STOP supported    :', npu.has_stop)

## 2. Scalar core: sort 64 words in DDR

The program is position independent (`auipc`), so the same instruction words run in the reference ISS at address 0 and on the board at the buffer's physical address. Data sits at byte offset 0x400 of the same buffer; the program sorts it in place and stores sum, xor, min and max after it. `npu.run` flushes the subsystem's caches before START and after the halt, so the ARM sees the results.

In [ ]:
N = 64
DATA_OFF = 0x400
SOURCE = f'''
start:
    auipc s0, 0
    addi  s0, s0, {DATA_OFF}      # s0 = data base
    mv    a0, s0
    li    t1, {N}
    li    s2, 0
    li    s3, 0
sum:
    lw    t2, 0(a0)
    add   s2, s2, t2
    xor   s3, s3, t2
    addi  a0, a0, 4
    addi  t1, t1, -1
    bnez  t1, sum
    li    a1, {N - 1}
outer:
    mv    a2, s0
    mv    a3, a1
inner:
    lw    t2, 0(a2)
    lw    t3, 4(a2)
    ble   t2, t3, noswap
    sw    t3, 0(a2)
    sw    t2, 4(a2)
noswap:
    addi  a2, a2, 4
    addi  a3, a3, -1
    bnez  a3, inner
    addi  a1, a1, -1
    bnez  a1, outer
    sw    s2, {4 * N}(s0)
    sw    s3, {4 * N + 4}(s0)
    lw    t2, 0(s0)
    sw    t2, {4 * N + 8}(s0)
    lw    t2, {4 * N - 4}(s0)
    sw    t2, {4 * N + 12}(s0)
    ecall
'''
program = assemble(SOURCE, base=0).words
print(len(program), 'instructions')

rng = np.random.default_rng(136)
data = rng.integers(-(1 << 31), 1 << 31, N, dtype=np.int64).astype(np.int32)

# Reference: the ISS (also gives the instruction count) and numpy
iss = RV32ISS(mem_size=0x800)
iss.load_words({4 * i: w for i, w in enumerate(program)})
iss.mem[DATA_OFF:DATA_OFF + 4 * N] = data.astype('<i4').tobytes()
iss_result = iss.run(0)
expected = np.concatenate([np.sort(data).view(np.uint32),
                           np.array([int(data.astype(np.int64).sum()) & 0xFFFFFFFF,
                                     int(np.bitwise_xor.reduce(data.view(np.uint32))),
                                     np.sort(data)[0].view(np.uint32), np.sort(data)[-1].view(np.uint32)],
                                    dtype=np.uint32)])
print('ISS:', iss_result.status, 'instructions:', iss_result.instret)

In [ ]:
buf = npu.allocate(0x800 // 4)               # 2 KB contiguous DDR buffer, uint32 words
words = np.asarray(buf)
words[:] = 0
words[DATA_OFF // 4:DATA_OFF // 4 + N] = data.view(np.uint32)
prog = npu.load_program(program, buf)       # writes the code at word 0 and flushes the ARM cache

before = npu.counters()
result = npu.run(prog.phys_addr, timeout=5.0)
buf.invalidate()                             # the subsystem wrote DDR behind the ARM cache
delta = npu.counters().delta(before)

got = words[DATA_OFF // 4:DATA_OFF // 4 + N + 4].copy()
assert np.array_equal(got, expected), 'scalar result differs from numpy'
assert result.instret == iss_result.instret
print(f'PASS  {result.instret} instructions in {result.cycles} cycles, CPI {result.cpi:.2f}, '
      f'{result.cycles / 100:.0f} us at 100 MHz')
print('cache hit rates:', {k: (None if v is None else round(v, 4)) for k, v in delta.hit_rates().items()})

## 3. Vector unit: INT8 saturating add and FP16 a*x+y

`VectorOp` keeps a reference kernel (from `src/runtime/hexagon.py`) and its DMA buffers resident; each call copies the operands in, DMAs them into the TCM, runs the kernel on the scalar core (which dispatches the vector commands), and DMAs the result back. Results must be bit-exact with numpy:

- INT8: `clip(x + y, -128, 127)` per lane;
- FP16: `a * x` rounded once to binary16, then `+ y` rounded once; any NaN result is the canonical `0x7E00`.

In [ ]:
ELEMENTS = 512

def int8_reference(x, y):
    return np.clip(x.astype(np.int16) + y.astype(np.int16), -128, 127).astype(np.int8)

def fp16_reference(a, x, y):
    a = np.float16(a)
    with np.errstate(all='ignore'):
        prod = (np.float32(a) * x.astype(np.float32)).astype(np.float16)
        out = (prod.astype(np.float64) + y.astype(np.float64)).astype(np.float16)
    bits = out.view(np.uint16).copy()
    bits[((bits & 0x7C00) == 0x7C00) & ((bits & 0x03FF) != 0)] = 0x7E00
    return bits.view(np.float16)

int8_add = VectorOp(npu, 'int8', ELEMENTS)
fp16_axpy = VectorOp(npu, 'fp16', ELEMENTS)

x8 = rng.integers(-128, 128, ELEMENTS).astype(np.int8)
y8 = rng.integers(-128, 128, ELEMENTS).astype(np.int8)
xf = rng.uniform(-8, 8, ELEMENTS).astype(np.float16)
yf = rng.uniform(-8, 8, ELEMENTS).astype(np.float16)
a = -0.75

out8 = int8_add(x8, y8, stats=True)
outf = fp16_axpy(xf, yf, a, stats=True)
assert np.array_equal(out8, int8_reference(x8, y8)), 'INT8 mismatch'
assert np.array_equal(outf.view(np.uint16), fp16_reference(a, xf, yf).view(np.uint16)), 'FP16 mismatch'
print('PASS  INT8 and FP16 results are bit-exact with numpy')
print('first lanes  INT8:', x8[:6], '+', y8[:6], '->', out8[:6])
print('             FP16:', a, '*', xf[:4], '+', yf[:4], '->', outf[:4])

In [ ]:
def median_seconds(fn, repeat=20):
    samples = []
    for _ in range(repeat):
        t = time.perf_counter(); fn(); samples.append(time.perf_counter() - t)
    return float(np.median(samples))

rows = []
for name, op, args, ref in (('INT8 x + y', int8_add, (x8, y8), lambda: int8_reference(x8, y8)),
                            ('FP16 a*x + y', fp16_axpy, (xf, yf, a), lambda: fp16_reference(a, xf, yf))):
    op(*args)                                     # warm: the steady state skips flushes and checks
    npu_s = median_seconds(lambda: op(*args))
    cycles = op.last_cycles
    np_s = median_seconds(ref)
    rows.append((name, cycles, cycles / 100, npu_s * 1e6, np_s * 1e6))

print(f"{'kernel':14s} {'cycles':>7s} {'kernel us':>10s} {'call us':>9s} {'numpy us':>9s}")
for name, cycles, kus, cus, nus in rows:
    print(f'{name:14s} {cycles:7d} {kus:10.1f} {cus:9.0f} {nus:9.0f}')
print('kernel = vector work on the NPU at 100 MHz; call = the whole Python call incl. DMA and ARM cache maintenance')

## 4. STOP: interrupt a program that never ends, then resume it

The program counts forever and stores the count into MAILBOX2. CONTROL.STOP halts the core at the next instruction boundary (VERSION >= 0x00010001); `fault_pc` is the first instruction not executed, and starting there continues with the registers intact.

In [ ]:
LOOP = '''
    lui   s0, 0x80000          # the subsystem's registers through the scalar MMIO window
    li    t0, 0
loop:
    addi  t0, t0, 1
    sw    t0, 0x58(s0)         # MAILBOX2 = count
    j     loop
'''
loop_prog = npu.load_program(assemble(LOOP, base=0).words)
npu.mailbox[2] = 0
npu.flush()                    # the caches must not hold an older program at this address
npu.start(loop_prog.phys_addr)
time.sleep(0.05)
first = npu.stop()
count1 = npu.mailbox[2]
print(f'stopped: {first.stopped}, resume pc 0x{first.fault_pc:08x}, count {count1}, {first.instret} instructions')

npu.start(first.fault_pc)      # resume where it stopped
time.sleep(0.05)
second = npu.stop()
count2 = npu.mailbox[2]
print(f'stopped: {second.stopped}, count {count2} (continued from {count1})')
assert first.stopped and second.stopped and count2 > count1

## 5. SmolLM2 on the same overlay

The SmolLM2-135M-Instruct package (W8A8 + AWQ) runs its GEMMs on the tensor accelerator through the ISA front end, next to the subsystem. Loading takes about 20 s and 24 tokens about 25 s.

In [ ]:
package = next((p for p in (here / 'export' / 'smollm', root / 'export' / 'smollm') if (p / 'manifest.json').is_file()), None)
if package is None:
    print('no export/smollm package here: skip this cell or run exp/1011_hexagon_npu/export_smollm.sh')
else:
    from src.inference.engine import LLMEngine
    t = time.perf_counter()
    engine = LLMEngine(package, 'pynq', overlay=overlay)
    print(f'loaded in {time.perf_counter() - t:.1f} s')
    ids, text, stats = engine.generate('What is the capital of France?', 24, chat=True)
    print(text)
    print({k: v for k, v in stats.as_dict().items() if k in ('prefill_tok_per_s', 'decode_tok_per_s', 'npu')})

## Clean up

In [ ]:
for op in (int8_add, fp16_axpy):
    op.free()
for b in (buf, loop_prog.buffer):
    b.freebuffer()
print('done')